# 06 — Reconstrucción de geometrías MT para QGIS

Este notebook reconstruye geometrías simples desde los datos CAD crudos cargados en PostGIS y publica capas iniciales en el esquema `gis`, listas para ser abiertas desde QGIS.

La reconstrucción se limita al servicio de media tensión de Montecarlo y trabaja con geometrías `POINT` y `LINESTRING`. A partir de la validación contra coordenadas transformadas a `EPSG:4326`, las geometrías se publican con `EPSG:32721` (`WGS 84 / UTM zone 21S`).

## 1. Objetivo y alcance

La meta es producir capas PostGIS consumibles por QGIS a partir de `crudo.tmp_shapefile` y, cuando esté disponible, enriquecer la trazabilidad con `crudo.objetos_red`.

| Capa CAD | Geometría esperada | Tabla final |
|---|---|---|
| `.00.MT_Cables` | `LINESTRING` | `gis.mt_cables` |
| `.00.MT_Postes` | `POINT` | `gis.mt_postes` |
| `.00.MT_Elementos` | `POINT` | `gis.mt_elementos` |
| `.00.MT_Seccionadores` | `POINT` | `gis.mt_seccionadores` |
| `.00.SET` | `POINT` | `gis.set` |

Quedan fuera de esta primera reconstrucción: transformadores, baja tensión, suministros, alumbrado, reclamos, catastro completo y textos como `.00.SET_Textos`.

## 2. Preparación de conexión

Usamos los mismos valores locales definidos para el laboratorio. Las variables de entorno permiten ajustar la conexión sin modificar la notebook.

In [11]:
# pathlib permite ubicar la raíz del proyecto aunque Jupyter se abra desde otra carpeta.
from pathlib import Path

# os permite leer variables de entorno para parametrizar la conexión local a PostGIS.
import os

# re ayuda a normalizar nombres de capas y claves CAD sin depender de mayúsculas o espacios.
import re

# datetime permite dejar una marca temporal legible en la auditoría de la reconstrucción.
from datetime import datetime

# psycopg es el driver moderno de PostgreSQL usado por el proyecto.
# El import defensivo evita un error oscuro si el kernel no tiene instaladas las dependencias.
try:
    import psycopg
    from psycopg import sql
    PSYCOPG_DISPONIBLE = True
except ModuleNotFoundError:
    psycopg = None
    sql = None
    PSYCOPG_DISPONIBLE = False
    print('No está instalado el paquete psycopg en este kernel.')
    print('Ejecutar desde la raíz del proyecto: python3 -m pip install -r requirements.txt')
    print('Después de instalar, reiniciar el kernel de Jupyter y volver a ejecutar la notebook.')

# Buscamos la raíz del proyecto subiendo hasta encontrar archivos propios del repo.
# No dependemos del nombre de la carpeta para que funcione aunque el repo se renombre.
RAIZ = Path.cwd()
while RAIZ.parent != RAIZ:
    if (RAIZ / 'docker-compose.yml').exists() and (RAIZ / 'scripts' / 'gis_sqlserver.sh').exists():
        break
    RAIZ = RAIZ.parent

# Cargamos .env local sin pisar variables ya definidas por el entorno.
def cargar_env_local() -> None:
    env_path = RAIZ / '.env'
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

cargar_env_local()

# Estos valores coinciden con el servicio local de PostGIS del laboratorio.
DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST', 'localhost'),
    'port': int(os.getenv('POSTGRES_PORT', '5432')),
    'dbname': os.getenv('POSTGRES_DB', 'ceml_gis'),
    'user': os.getenv('POSTGRES_USER', 'ceml'),
    'password': os.getenv('POSTGRES_PASSWORD', 'ceml_admin_2026'),
}

# Usamos EPSG:32721 porque las coordenadas CAD están en metros y transforman correctamente a Montecarlo, Misiones.
# QGIS puede reproyectar estas capas a EPSG:4326 o EPSG:3857 para verlas sobre mapas base.
SRID_PUBLICACION = 32721

# Esta función corta con una instrucción clara si falta el driver de PostgreSQL.
def exigir_psycopg() -> None:
    if not PSYCOPG_DISPONIBLE:
        raise RuntimeError(
            'Falta instalar psycopg en este kernel. '
            'Ejecutar: python -m pip install -r requirements.txt, '
            'reiniciar el kernel y volver a correr la notebook.'
        )

# Mostramos el contexto para que el estudiante confirme que está trabajando sobre la base correcta.
print('Raíz del proyecto:', RAIZ)
print('Base local:', DB_CONFIG['dbname'])
print('Host local:', DB_CONFIG['host'])
print('Puerto local:', DB_CONFIG['port'])
print('Usuario local:', DB_CONFIG['user'])
print('SRID de publicación:', SRID_PUBLICACION)


Raíz del proyecto: /home/jovyan/work
Base local: ceml_gis
Host local: postgis
Puerto local: 5432
Usuario local: ceml
SRID de publicación: 32721


## 3. Verificación de PostGIS y prerrequisitos

Antes de reconstruir geometrías, confirmamos que existen los esquemas y tablas creados por el notebook 05. Si faltan datos crudos, esta sección informa qué notebooks deben ejecutarse primero.

In [12]:
# Definimos esquemas requeridos y tablas crudas del piloto.
# Solo tmp_shapefile es obligatorio para reconstruir geometrías CAD; las demás tablas enriquecen atributos.
ESQUEMAS_REQUERIDOS = ['crudo', 'depuracion', 'gis', 'auditoria']
TABLAS_CRUDAS_OBLIGATORIAS = [
    ('crudo', 'tmp_shapefile'),
]
TABLAS_CRUDAS_OPCIONALES = [
    ('crudo', 'objetos_red'),
    ('crudo', 'set'),
    ('crudo', 'seccionadores'),
    ('crudo', 'mt_cables'),
]
TABLAS_CRUDAS_A_REVISAR = TABLAS_CRUDAS_OBLIGATORIAS + TABLAS_CRUDAS_OPCIONALES

# Esta función consulta si una tabla existe sin depender de errores posteriores de SQL.
def existe_tabla(cur, esquema: str, tabla: str) -> bool:
    cur.execute(
        """
        SELECT EXISTS (
            SELECT 1
            FROM information_schema.tables
            WHERE table_schema = %s AND table_name = %s
        );
        """,
        (esquema, tabla),
    )
    return cur.fetchone()[0]

# Esta función cuenta filas con identificadores seguros para evitar concatenar SQL manualmente.
def contar_filas(cur, esquema: str, tabla: str) -> int:
    cur.execute(sql.SQL('SELECT count(*) FROM {}.{};').format(sql.Identifier(esquema), sql.Identifier(tabla)))
    return cur.fetchone()[0]

# Validamos conexión, extensión PostGIS y disponibilidad mínima de tablas crudas.
exigir_psycopg()
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute('CREATE EXTENSION IF NOT EXISTS postgis;')
        cur.execute('SELECT PostGIS_Full_Version();')
        version_postgis = cur.fetchone()[0]

        cur.execute(
            """
            SELECT schema_name
            FROM information_schema.schemata
            WHERE schema_name = ANY(%s)
            ORDER BY schema_name;
            """,
            (ESQUEMAS_REQUERIDOS,),
        )
        esquemas_presentes = {fila[0] for fila in cur.fetchall()}
        esquemas_faltantes = sorted(set(ESQUEMAS_REQUERIDOS) - esquemas_presentes)

        estado_tablas = []
        for esquema, tabla in TABLAS_CRUDAS_A_REVISAR:
            existe = existe_tabla(cur, esquema, tabla)
            filas = contar_filas(cur, esquema, tabla) if existe else 0
            estado_tablas.append({'tabla': f'{esquema}.{tabla}', 'existe': existe, 'filas': filas})

# Informamos primero lo que sí está disponible para que el diagnóstico sea transparente.
print('Conexión a PostGIS verificada correctamente.')
print('Versión PostGIS:', version_postgis)
for item in estado_tablas:
    print(f"{item['tabla']}: existe={item['existe']} filas={item['filas']}")

# Separamos faltantes obligatorios de opcionales para no bloquear capas que sí pueden reconstruirse.
nombres_obligatorios = {f'{esquema}.{tabla}' for esquema, tabla in TABLAS_CRUDAS_OBLIGATORIAS}
tablas_obligatorias_faltantes = [item['tabla'] for item in estado_tablas if item['tabla'] in nombres_obligatorios and not item['existe']]
tablas_obligatorias_vacias = [item['tabla'] for item in estado_tablas if item['tabla'] in nombres_obligatorios and item['existe'] and item['filas'] == 0]
tablas_opcionales_faltantes = [item['tabla'] for item in estado_tablas if item['tabla'] not in nombres_obligatorios and not item['existe']]
tablas_opcionales_vacias = [item['tabla'] for item in estado_tablas if item['tabla'] not in nombres_obligatorios and item['existe'] and item['filas'] == 0]
if esquemas_faltantes or tablas_obligatorias_faltantes or tablas_obligatorias_vacias:
    raise RuntimeError(
        'Faltan prerrequisitos para reconstruir geometrías. '
        f'Esquemas faltantes: {esquemas_faltantes}. '
        f'Tablas obligatorias faltantes: {tablas_obligatorias_faltantes}. '
        f'Tablas obligatorias vacías: {tablas_obligatorias_vacias}. '
        'Ejecutar primero el notebook 01 para exportar TSV y luego el notebook 05 para crear/cargar PostGIS.'
    )

# Las tablas opcionales no detienen la reconstrucción; solo limitan enriquecimiento o validaciones.
if tablas_opcionales_faltantes or tablas_opcionales_vacias:
    print('Advertencia: algunas tablas opcionales no están disponibles.')
    print('Opcionales faltantes:', tablas_opcionales_faltantes)
    print('Opcionales vacías:', tablas_opcionales_vacias)
    print('Se continúa con las geometrías reconstruibles desde crudo.tmp_shapefile.')


Conexión a PostGIS verificada correctamente.
Versión PostGIS: POSTGIS="3.4.3 e365945" [EXTENSION] PGSQL="160" GEOS="3.9.0-CAPI-1.16.2" PROJ="7.2.1 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=/var/lib/postgresql/.local/share/proj DATABASE_PATH=/usr/share/proj/proj.db" LIBXML="2.9.10" LIBJSON="0.15" LIBPROTOBUF="1.3.3" WAGYU="0.5.0 (Internal)" TOPOLOGY
crudo.tmp_shapefile: existe=True filas=125688
crudo.objetos_red: existe=True filas=69134
crudo.set: existe=False filas=0
crudo.seccionadores: existe=True filas=1063
crudo.mt_cables: existe=True filas=40
Advertencia: algunas tablas opcionales no están disponibles.
Opcionales faltantes: ['crudo.set']
Opcionales vacías: []
Se continúa con las geometrías reconstruibles desde crudo.tmp_shapefile.


## 4. Lectura y normalización de columnas

Los TSV crudos pueden conservar nombres de SQL Server con mayúsculas, minúsculas o variantes. Por eso resolvemos columnas de forma insensible a mayúsculas y normalizamos las claves `idcad/IDCAD` y `coop/COOP`.

In [13]:
# Esta función simplifica nombres para comparar columnas sin depender del casing original.
def normalizar_nombre_columna(nombre: str) -> str:
    return re.sub(r'[^a-z0-9]', '', nombre.lower())

# Esta función devuelve el nombre real de una columna a partir de una lista de candidatos lógicos.
def buscar_columna(columnas: list[str], candidatos: list[str], requerida: bool = True) -> str | None:
    indice = {normalizar_nombre_columna(columna): columna for columna in columnas}
    for candidato in candidatos:
        columna = indice.get(normalizar_nombre_columna(candidato))
        if columna:
            return columna
    if requerida:
        raise RuntimeError(f"No se encontró ninguna columna compatible con: {candidatos}")
    return None

# Esta función lee los nombres reales de columnas desde information_schema.
def obtener_columnas(cur, esquema: str, tabla: str) -> list[str]:
    cur.execute(
        """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position;
        """,
        (esquema, tabla),
    )
    return [fila[0] for fila in cur.fetchall()]

# Confirmamos qué columnas usaremos para capa, geometría cruda y claves de cruce.
with psycopg.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        columnas_tmp = obtener_columnas(cur, 'crudo', 'tmp_shapefile')
        columnas_objetos = obtener_columnas(cur, 'crudo', 'objetos_red')

col_tmp_datos = buscar_columna(columnas_tmp, ['Datos_Objeto', 'datos_objeto', 'datosobjeto'])
col_tmp_capa = buscar_columna(columnas_tmp, ['AC_CAPA', 'Layer', 'Capa', 'Nombre_Capa', 'nombre_capa'], requerida=False)
col_tmp_idcad = buscar_columna(columnas_tmp, ['idcad', 'IDCAD'], requerida=False)
col_tmp_coop = buscar_columna(columnas_tmp, ['coop', 'COOP'], requerida=False)
col_obj_idcad = buscar_columna(columnas_objetos, ['idcad', 'IDCAD'], requerida=False)
col_obj_coop = buscar_columna(columnas_objetos, ['coop', 'COOP'], requerida=False)

# Mostramos el mapeo detectado para que el estudiante pueda auditarlo antes de reconstruir.
print('Columnas detectadas en crudo.tmp_shapefile:')
print('Datos_Objeto:', col_tmp_datos)
print('Capa:', col_tmp_capa)
print('IDCAD:', col_tmp_idcad)
print('COOP:', col_tmp_coop)
print('Columnas detectadas en crudo.objetos_red:')
print('IDCAD:', col_obj_idcad)
print('COOP:', col_obj_coop)


Columnas detectadas en crudo.tmp_shapefile:
Datos_Objeto: Datos_Objeto
Capa: AC_CAPA
IDCAD: IDCAD
COOP: COOP
Columnas detectadas en crudo.objetos_red:
IDCAD: idcad
COOP: coop


## 5. Cómo interpretar `Datos_Objeto`

La columna `Datos_Objeto` contiene texto con estructura similar a DXF. Los tokens están separados por `§`. En DXF, varios atributos se expresan como pares `código -> valor`. Para esta etapa nos interesan estos códigos:

| Código | Uso en esta notebook |
|---|---|
| `10` | Coordenada X principal o X de vértice |
| `20` | Coordenada Y principal o Y de vértice |
| `11` | Coordenada X final en entidades `LINE` |
| `21` | Coordenada Y final en entidades `LINE` |
| `70` | Bandera de cierre en `LWPOLYLINE` cuando está disponible |

Este parser es intencionalmente acotado: reconoce `INSERT`, `CIRCLE`, `LINE` y `LWPOLYLINE`. Las polilíneas cerradas se detectan, pero no se publican como polígonos en esta primera etapa MT.

In [14]:
# Definimos las entidades mínimas que el laboratorio necesita interpretar.
ENTIDADES_SOPORTADAS = {'INSERT', 'CIRCLE', 'LINE', 'LWPOLYLINE'}

# Esta función separa los tokens DXF-like descartando vacíos producidos por separadores repetidos.
def tokenizar_datos_objeto(valor: object) -> list[str]:
    if valor is None:
        return []
    return [token.strip() for token in str(valor).split('§') if token.strip()]

# Esta función separa un token DXF-like como '10,722060.1,7059563.7,0' en código y valores.
def dividir_token(token: str) -> tuple[str, list[str]]:
    partes = [parte.strip() for parte in str(token).split(',')]
    codigo = partes[0] if partes else ''
    valores = partes[1:] if len(partes) > 1 else []
    return codigo, valores

# Esta función identifica el tipo de entidad buscando el código 0 o tokens conocidos.
def identificar_entidad(tokens: list[str]) -> str | None:
    for token in tokens:
        codigo, valores = dividir_token(token)
        entidad = valores[0].upper() if codigo == '0' and valores else token.upper()
        if entidad in ENTIDADES_SOPORTADAS:
            return entidad
    return None

# Esta función extrae todos los grupos de valores asociados a un código DXF-like.
def grupos_codigo(tokens: list[str], codigo_buscado: str) -> list[list[str]]:
    grupos = []
    for token in tokens:
        codigo, valores = dividir_token(token)
        if codigo == codigo_buscado:
            grupos.append(valores)
    return grupos

# Esta función mantiene compatibilidad con tokens DXF separados como código -> valor.
def valores_codigo(tokens: list[str], codigo: str) -> list[str]:
    valores = []
    for grupo in grupos_codigo(tokens, codigo):
        if grupo:
            valores.append(grupo[0])
    for indice, token in enumerate(tokens[:-1]):
        if token == codigo:
            valores.append(tokens[indice + 1])
    return valores

# Esta función convierte números escritos como texto, contemplando coma decimal si apareciera.
def convertir_float(valor: str) -> float | None:
    try:
        return float(str(valor).replace(',', '.'))
    except (TypeError, ValueError):
        return None

# Esta función obtiene una coordenada XY desde un grupo tipo ['x', 'y', 'z'].
def punto_desde_grupo(grupo: list[str]) -> tuple[float, float] | None:
    if len(grupo) < 2:
        return None
    x = convertir_float(grupo[0])
    y = convertir_float(grupo[1])
    if x is None or y is None:
        return None
    return (x, y)

# Esta función arma pares X/Y desde grupos 10,x,y,z o desde listas repetidas 10/20.
def pares_xy(tokens: list[str]) -> list[tuple[float, float]]:
    puntos_compactos = []
    for grupo in grupos_codigo(tokens, '10'):
        punto = punto_desde_grupo(grupo)
        if punto is not None:
            puntos_compactos.append(punto)
    if puntos_compactos:
        return puntos_compactos

    xs = [convertir_float(valor) for valor in valores_codigo(tokens, '10')]
    ys = [convertir_float(valor) for valor in valores_codigo(tokens, '20')]
    pares = []
    for x, y in zip(xs, ys):
        if x is not None and y is not None:
            pares.append((x, y))
    return pares

# Esta función formatea coordenadas para WKT sin agregar precisión artificial.
def formatear_numero(valor: float) -> str:
    return format(valor, '.12g')

# Esta función construye WKT POINT a partir de una coordenada.
def wkt_point(punto: tuple[float, float]) -> str:
    x, y = punto
    return f"POINT({formatear_numero(x)} {formatear_numero(y)})"

# Esta función construye WKT LINESTRING a partir de dos o más coordenadas.
def wkt_linestring(puntos: list[tuple[float, float]]) -> str:
    coordenadas = ', '.join(f"{formatear_numero(x)} {formatear_numero(y)}" for x, y in puntos)
    return f'LINESTRING({coordenadas})'

# Esta función detecta si una LWPOLYLINE está cerrada por bandera 70 o por repetir primer y último vértice.
def es_polilinea_cerrada(tokens: list[str], puntos: list[tuple[float, float]]) -> bool:
    banderas = [convertir_float(valor) for valor in valores_codigo(tokens, '70')]
    cerrada_por_bandera = any(int(bandera) & 1 == 1 for bandera in banderas if bandera is not None)
    cerrada_por_vertices = len(puntos) > 2 and puntos[0] == puntos[-1]
    return cerrada_por_bandera or cerrada_por_vertices

# Esta función reconstruye solo candidatos POINT y LINESTRING válidos para la etapa MT.
def reconstruir_geometria(datos_objeto: object) -> dict[str, object]:
    tokens = tokenizar_datos_objeto(datos_objeto)
    entidad = identificar_entidad(tokens)
    if not tokens:
        return {'entidad': None, 'tipo_geometria': None, 'wkt': None, 'estado': 'rechazado', 'notas': 'Datos_Objeto vacío'}
    if entidad is None:
        return {'entidad': None, 'tipo_geometria': None, 'wkt': None, 'estado': 'rechazado', 'notas': 'Entidad no soportada o no identificada'}

    puntos = pares_xy(tokens)
    if entidad in {'INSERT', 'CIRCLE'}:
        if puntos:
            return {'entidad': entidad, 'tipo_geometria': 'POINT', 'wkt': wkt_point(puntos[0]), 'estado': 'ok', 'notas': ''}
        return {'entidad': entidad, 'tipo_geometria': None, 'wkt': None, 'estado': 'rechazado', 'notas': 'Faltan coordenadas 10/20 para punto'}

    if entidad == 'LINE':
        inicio = punto_desde_grupo(grupos_codigo(tokens, '10')[0]) if grupos_codigo(tokens, '10') else None
        fin = punto_desde_grupo(grupos_codigo(tokens, '11')[0]) if grupos_codigo(tokens, '11') else None
        if inicio is not None and fin is not None:
            return {'entidad': entidad, 'tipo_geometria': 'LINESTRING', 'wkt': wkt_linestring([inicio, fin]), 'estado': 'ok', 'notas': ''}
        x1 = convertir_float(valores_codigo(tokens, '10')[0]) if valores_codigo(tokens, '10') else None
        y1 = convertir_float(valores_codigo(tokens, '20')[0]) if valores_codigo(tokens, '20') else None
        x2 = convertir_float(valores_codigo(tokens, '11')[0]) if valores_codigo(tokens, '11') else None
        y2 = convertir_float(valores_codigo(tokens, '21')[0]) if valores_codigo(tokens, '21') else None
        if None not in (x1, y1, x2, y2):
            return {'entidad': entidad, 'tipo_geometria': 'LINESTRING', 'wkt': wkt_linestring([(x1, y1), (x2, y2)]), 'estado': 'ok', 'notas': ''}
        return {'entidad': entidad, 'tipo_geometria': None, 'wkt': None, 'estado': 'rechazado', 'notas': 'Faltan coordenadas 10/20/11/21 para línea'}

    if entidad == 'LWPOLYLINE':
        cerrada = es_polilinea_cerrada(tokens, puntos)
        if cerrada:
            return {'entidad': entidad, 'tipo_geometria': None, 'wkt': None, 'estado': 'omitido', 'notas': 'Polilínea cerrada detectada; no se publica como polígono en esta etapa MT'}
        if len(puntos) >= 2:
            return {'entidad': entidad, 'tipo_geometria': 'LINESTRING', 'wkt': wkt_linestring(puntos), 'estado': 'ok', 'notas': ''}
        return {'entidad': entidad, 'tipo_geometria': None, 'wkt': None, 'estado': 'rechazado', 'notas': 'LWPOLYLINE sin dos vértices válidos'}

    return {'entidad': entidad, 'tipo_geometria': None, 'wkt': None, 'estado': 'rechazado', 'notas': 'Entidad fuera de alcance'}

# Dejamos una confirmación simple de que las funciones están listas para usarse.
print('Funciones de reconstrucción preparadas para INSERT, CIRCLE, LINE y LWPOLYLINE.')


Funciones de reconstrucción preparadas para INSERT, CIRCLE, LINE y LWPOLYLINE.


## 6. Tabla de depuración

La tabla `depuracion.geometrias_cad_mt` conserva cada intento de reconstrucción. Esta etapa es importante porque permite auditar rechazos, capas omitidas y geometrías publicadas sin perder la fila original.

In [15]:
# Esta sentencia crea una tabla trazable para revisar resultados antes de publicar en gis.
SQL_DEPURACION = """
DROP TABLE IF EXISTS depuracion.geometrias_cad_mt CASCADE;
CREATE TABLE depuracion.geometrias_cad_mt (
    id bigserial PRIMARY KEY,
    tabla_origen text NOT NULL,
    fila_origen bigint NOT NULL,
    capa_origen text,
    tipo_entidad text,
    tipo_geometria text,
    wkt text,
    idcad text,
    coop text,
    estado_parseo text NOT NULL,
    notas text NOT NULL DEFAULT '',
    creado_en timestamptz NOT NULL DEFAULT now()
);
"""

# Reemplazamos la tabla de depuración para que cada ejecución sea reproducible en clase.
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(SQL_DEPURACION)

# Informamos el destino donde quedarán todos los candidatos y rechazos.
print('Tabla de depuración preparada: depuracion.geometrias_cad_mt')


Tabla de depuración preparada: depuracion.geometrias_cad_mt


## 7. Reconstrucción controlada desde `crudo.tmp_shapefile`

Procesamos las filas crudas en Python para mantener visible la lógica didáctica del parser. La clave `idcad + coop` se normaliza con `trim` y mayúsculas para facilitar cruces posteriores.

In [16]:
# Esta función normaliza claves de unión quitando espacios y unificando mayúsculas.
def normalizar_clave(valor: object) -> str | None:
    if valor is None:
        return None
    texto = str(valor).strip().upper()
    return texto or None

# Esta función normaliza capas para comparar contra el mapeo acordado.
def normalizar_capa(valor: object) -> str:
    return str(valor or '').strip()

# Definimos expresiones SQL seguras para leer columnas reales aunque tengan casing original.
selector_fila = sql.SQL('ctid::text')
selector_datos = sql.Identifier(col_tmp_datos)
selector_capa = sql.Identifier(col_tmp_capa) if col_tmp_capa else sql.SQL('NULL')
selector_idcad = sql.Identifier(col_tmp_idcad) if col_tmp_idcad else sql.SQL('NULL')
selector_coop = sql.Identifier(col_tmp_coop) if col_tmp_coop else sql.SQL('NULL')

# Leemos las filas crudas y guardamos cada resultado en la tabla de depuración.
insertados = 0
with psycopg.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        consulta = sql.SQL('SELECT {}, {}, {}, {}, {} FROM crudo.tmp_shapefile;').format(
            selector_fila, selector_datos, selector_capa, selector_idcad, selector_coop
        )
        cur.execute(consulta)
        filas = cur.fetchall()

        for indice, (fila_origen, datos_objeto, capa, idcad, coop) in enumerate(filas, start=1):
            resultado = reconstruir_geometria(datos_objeto)
            cur.execute(
                """
                INSERT INTO depuracion.geometrias_cad_mt (
                    tabla_origen, fila_origen, capa_origen, tipo_entidad, tipo_geometria,
                    wkt, idcad, coop, estado_parseo, notas
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
                """,
                (
                    'crudo.tmp_shapefile',
                    indice,
                    normalizar_capa(capa),
                    resultado['entidad'],
                    resultado['tipo_geometria'],
                    resultado['wkt'],
                    normalizar_clave(idcad),
                    normalizar_clave(coop),
                    resultado['estado'],
                    resultado['notas'],
                ),
            )
            insertados += 1

    conn.commit()

# Mostramos un resumen inicial de filas procesadas.
print('Filas procesadas desde crudo.tmp_shapefile:', insertados)


Filas procesadas desde crudo.tmp_shapefile: 125688


## 8. Cruce opcional con `crudo.objetos_red`

Cuando `crudo.tmp_shapefile` y `crudo.objetos_red` tienen las columnas `IDCAD` y `COOP`, calculamos la cobertura del cruce normalizado. Esta validación no bloquea la publicación, pero ayuda a medir cuánta geometría CAD puede vincularse con objetos de red.

In [17]:
# Solo calculamos cobertura si ambos orígenes tienen las claves necesarias.
columnas_join_disponibles = all([col_tmp_idcad, col_tmp_coop, col_obj_idcad, col_obj_coop])

# El cruce usa las claves ya normalizadas en depuración contra una vista lógica de objetos_red.
if columnas_join_disponibles:
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(
                sql.SQL(
                    """
                    WITH objetos AS (
                        SELECT DISTINCT
                            upper(trim({})) AS idcad,
                            upper(trim({})) AS coop
                        FROM crudo.objetos_red
                        WHERE nullif(trim({}), '') IS NOT NULL
                          AND nullif(trim({}), '') IS NOT NULL
                    ), cobertura AS (
                        SELECT
                            count(*) FILTER (WHERE d.idcad IS NOT NULL AND d.coop IS NOT NULL) AS con_clave,
                            count(*) FILTER (WHERE o.idcad IS NOT NULL) AS con_objeto_red
                        FROM depuracion.geometrias_cad_mt d
                        LEFT JOIN objetos o
                          ON o.idcad = d.idcad AND o.coop = d.coop
                    )
                    SELECT con_clave, con_objeto_red
                    FROM cobertura;
                    """
                ).format(
                    sql.Identifier(col_obj_idcad),
                    sql.Identifier(col_obj_coop),
                    sql.Identifier(col_obj_idcad),
                    sql.Identifier(col_obj_coop),
                )
            )
            con_clave, con_objeto_red = cur.fetchone()
            porcentaje = (con_objeto_red / con_clave * 100) if con_clave else 0

    # Mostramos el indicador de cobertura como control de calidad, no como filtro de publicación.
    print('Filas CAD con clave normalizada:', con_clave)
    print('Filas CAD con coincidencia en objetos_red:', con_objeto_red)
    print(f'Cobertura aproximada del cruce: {porcentaje:.2f}%')
else:
    # Si faltan columnas, seguimos con geometría porque el cruce es informativo en esta etapa.
    print('No se calcula cobertura con objetos_red porque faltan columnas IDCAD/COOP en alguno de los orígenes.')


Filas CAD con clave normalizada: 125252
Filas CAD con coincidencia en objetos_red: 47465
Cobertura aproximada del cruce: 37.90%


## 9. Publicación de capas finales en `GIS`

Las tablas finales se crean con geometría PostGIS y SRID `32721`. Este CRS corresponde a `WGS 84 / UTM zone 21S`, validado al transformar puntos de muestra a `EPSG:4326` y verificar que caen sobre Montecarlo, Misiones.

In [18]:
# Este mapeo define exactamente qué capas CAD se publican y con qué tipo geométrico.
CAPAS_PUBLICACION = [
    {
        'capa': '.00.MT_Cables',
        'tabla': 'mt_cables',
        'tipo': 'LINESTRING',
    },
    {
        'capa': '.00.MT_Postes',
        'tabla': 'mt_postes',
        'tipo': 'POINT',
    },
    {
        'capa': '.00.MT_Elementos',
        'tabla': 'mt_elementos',
        'tipo': 'POINT',
    },
    {
        'capa': '.00.MT_Seccionadores',
        'tabla': 'mt_seccionadores',
        'tipo': 'POINT',
    },
    {
        'capa': '.00.SET',
        'tabla': 'set',
        'tipo': 'POINT',
    },
]

# Esta función crea una tabla final homogénea para QGIS.
def crear_tabla_final(cur, tabla: str, tipo_geometria: str) -> None:
    cur.execute(sql.SQL('DROP TABLE IF EXISTS gis.{};').format(sql.Identifier(tabla)))
    cur.execute(
        sql.SQL(
            """
            CREATE TABLE gis.{} (
                id bigserial PRIMARY KEY,
                depuracion_id bigint NOT NULL REFERENCES depuracion.geometrias_cad_mt(id),
                capa_origen text NOT NULL,
                tipo_entidad text NOT NULL,
                idcad text,
                coop text,
                geom geometry({}, {}) NOT NULL
            );
            """
        ).format(sql.Identifier(tabla), sql.SQL(tipo_geometria), sql.Literal(SRID_PUBLICACION))
    )

# Creamos e insertamos cada capa final desde los candidatos válidos de depuración.
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
    with conn.cursor() as cur:
        for item in CAPAS_PUBLICACION:
            crear_tabla_final(cur, item['tabla'], item['tipo'])
            cur.execute(
                sql.SQL(
                    """
                    INSERT INTO gis.{} (depuracion_id, capa_origen, tipo_entidad, idcad, coop, geom)
                    SELECT
                        id,
                        capa_origen,
                        tipo_entidad,
                        idcad,
                        coop,
                        ST_SetSRID(ST_GeomFromText(wkt), {})::geometry({}, {}) AS geom
                    FROM depuracion.geometrias_cad_mt
                    WHERE estado_parseo = 'ok'
                      AND capa_origen = %s
                      AND tipo_geometria = %s
                      AND wkt IS NOT NULL
                      AND NOT ST_IsEmpty(ST_GeomFromText(wkt));
                    """
                ).format(
                    sql.Identifier(item['tabla']),
                    sql.Literal(SRID_PUBLICACION),
                    sql.SQL(item['tipo']),
                    sql.Literal(SRID_PUBLICACION),
                ),
                (item['capa'], item['tipo']),
            )
            cur.execute(sql.SQL('CREATE INDEX {} ON gis.{} USING gist (geom);').format(
                sql.Identifier(f"idx_gis_{item['tabla']}_geom"),
                sql.Identifier(item['tabla']),
            ))

# Informamos que las tablas finales fueron regeneradas.
print('Capas finales publicadas en el esquema gis con índices espaciales.')


Capas finales publicadas en el esquema gis con índices espaciales.


## 10. Auditoría de reconstrucción

La auditoría registra métricas de publicación y rechazo. Esto ayuda a explicar por qué una entidad no aparece en QGIS: puede estar fuera de alcance, tener geometría cerrada omitida o no contar con coordenadas suficientes.

In [19]:
# Creamos una tabla de auditoría específica para esta reconstrucción.
SQL_AUDITORIA_RECONSTRUCCION = """
CREATE TABLE IF NOT EXISTS auditoria.reconstruccion_mt (
    id bigserial PRIMARY KEY,
    fecha_ejecucion timestamptz NOT NULL DEFAULT now(),
    etapa text NOT NULL,
    capa text,
    tabla_destino text,
    filas integer NOT NULL DEFAULT 0,
    notas text NOT NULL DEFAULT ''
);
"""

# Insertamos métricas por estado de parseo y por tabla final publicada.
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute(SQL_AUDITORIA_RECONSTRUCCION)
        cur.execute(
            """
            INSERT INTO auditoria.reconstruccion_mt (etapa, capa, tabla_destino, filas, notas)
            SELECT
                'parseo' AS etapa,
                capa_origen,
                NULL AS tabla_destino,
                count(*)::integer AS filas,
                concat('estado=', estado_parseo, '; tipo=', coalesce(tipo_geometria, 'sin_geometria')) AS notas
            FROM depuracion.geometrias_cad_mt
            GROUP BY capa_origen, estado_parseo, tipo_geometria;
            """
        )
        for item in CAPAS_PUBLICACION:
            cur.execute(sql.SQL('SELECT count(*) FROM gis.{};').format(sql.Identifier(item['tabla'])))
            filas = cur.fetchone()[0]
            cur.execute(
                """
                INSERT INTO auditoria.reconstruccion_mt (etapa, capa, tabla_destino, filas, notas)
                VALUES ('publicacion', %s, %s, %s, %s);
                """,
                (item['capa'], f"gis.{item['tabla']}", filas, f"SRID {SRID_PUBLICACION}; validado contra EPSG:4326 para Montecarlo, Misiones"),
            )

# Dejamos una marca visual de cierre de auditoría para la clase.
print('Auditoría registrada en auditoria.reconstruccion_mt:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


Auditoría registrada en auditoria.reconstruccion_mt: 2026-06-06 21:35:40


## 11. Validaciones para revisar antes de abrir QGIS

Estas consultas verifican cantidad de registros, tipos geométricos, geometrías vacías o inválidas y una muestra de WKT/extensión. Si una capa tiene cero filas, revisar primero `depuracion.geometrias_cad_mt` y `auditoria.reconstruccion_mt`.

In [20]:
# Recorremos las capas finales y mostramos métricas mínimas para validar la publicación.
with psycopg.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        for item in CAPAS_PUBLICACION:
            tabla = item['tabla']
            cur.execute(sql.SQL(
                """
                SELECT
                    count(*) AS filas,
                    array_agg(DISTINCT GeometryType(geom)) AS tipos,
                    count(*) FILTER (WHERE ST_IsEmpty(geom)) AS vacias,
                    count(*) FILTER (WHERE NOT ST_IsValid(geom)) AS invalidas,
                    ST_AsText(ST_Extent(geom)) AS extension
                FROM gis.{};
                """
            ).format(sql.Identifier(tabla)))
            filas, tipos, vacias, invalidas, extension = cur.fetchone()
            print(f"gis.{tabla}: filas={filas}, tipos={tipos}, vacías={vacias}, inválidas={invalidas}, extensión={extension}")

            # Mostramos hasta tres WKT cortos para confirmar visualmente la geometría resultante.
            cur.execute(sql.SQL('SELECT ST_AsText(geom) FROM gis.{} LIMIT 3;').format(sql.Identifier(tabla)))
            muestras = [fila[0] for fila in cur.fetchall()]
            for muestra in muestras:
                print('  muestra:', muestra)


gis.mt_cables: filas=7157, tipos=['LINESTRING'], vacías=0, inválidas=14, extensión=POLYGON((707304 7025587.0086,707304 7073013.5745,746416.0731 7073013.5745,746416.0731 7025587.0086,707304 7025587.0086))
  muestra: LINESTRING(723707.1498 7059005.314,723721.4859 7059001.7654)
  muestra: LINESTRING(723724.8725 7059374.0841,723725.8465 7059408.1477)
  muestra: LINESTRING(723725.2279 7059341.4951,723724.8725 7059374.0841)
gis.mt_postes: filas=7016, tipos=['POINT'], vacías=0, inválidas=0, extensión=POLYGON((707304 7025587.0086,707304 7073013.5745,746416.0731 7073013.5745,746416.0731 7025587.0086,707304 7025587.0086))
  muestra: POINT(719720 7047675)
  muestra: POINT(721017.9151 7060233.2151)
  muestra: POINT(725461.54 7049079.3699)
gis.mt_elementos: filas=3062, tipos=['POINT'], vacías=0, inválidas=0, extensión=POLYGON((707295.6995 7025585.4377,707295.6995 7071891.7221,746420.8421 7071891.7221,746420.8421 7025585.4377,707295.6995 7025585.4377))
  muestra: POINT(723835.8492 7055733.7261)
  mu

## 12. Cierre de reconstrucción y publicación GIS

En esta notebook se reconstruyeron geometrías MT desde `crudo.tmp_shapefile`, se registraron métricas de auditoría y se publicaron capas iniciales en `gis` con SRID `32721` para revisar desde QGIS.

- Capas publicadas: `gis.mt_cables`, `gis.mt_postes`, `gis.mt_elementos`, `gis.mt_seccionadores` y `gis.set`.
- El siguiente paso es abrirlas en QGIS con la conexión local a `ceml_gis` y revisar visualmente cobertura, ubicación y entidades inválidas.